In [1]:
!pip install pandas scipy tqdm numpy


**required files**:
scorer.py,
baseline_random_guess.py,
mushroom.en-val.v2.jsonl

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

In [ ]:
path = "../task_data"

In [29]:
scorer = os.path.join(path, "participant_kit", "scorer.py")
baseline_random_guess = os.path.join(path, "participant_kit", "baseline_random_guess.py")
mushroom = os.path.join(path, "val", "mushroom.en-val.v2.jsonl")

In [ ]:
#mushroom

In [ ]:
#run basic ## changed "import scorer.py" to "import scorer" in original baseline_random_guess.py
!python {baseline_random_guess} {mushroom} --output_file baseline_random-predictions-for-en.jsonl
!cat baseline_random-predictions-for-en.jsonl

{"id":"val-en-1","lang":"EN","model_input":"What did Petra van Staveren win a gold medal for?","model_output_text":"Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China.","model_id":"tiiuae\/falcon-7b-instruct","soft_labels":[{"start":0,"end":1,"prob":0.5},{"start":1,"end":2,"prob":0.4},{"start":2,"end":3,"prob":0.1818181818},{"start":3,"end":4,"prob":0.2222222222},{"start":4,"end":5,"prob":0.3},{"start":5,"end":6,"prob":0.1111111111},{"start":6,"end":7,"prob":0.0},{"start":7,"end":8,"prob":0.1},{"start":8,"end":9,"prob":0.0},{"start":9,"end":10,"prob":1.0},{"start":10,"end":11,"prob":0.7},{"start":11,"end":12,"prob":0.0},{"start":12,"end":13,"prob":0.3},{"start":13,"end":14,"prob":0.5454545455},{"start":14,"end":15,"prob":0.3},{"start":15,"end":16,"prob":0.3},{"start":16,"end":17,"prob":0.3},{"start":17,"end":18,"prob":0.1111111111},{"start":18,"end":19,"prob":0.1818181818},{"start":19,"end":20,"prob":0.1},{"start":20,"end":21,"prob":0.4444444444},{"star

In [ ]:
#run with bootstraping
!python {baseline_random_guess} {mushroom} --output_file baseline_random-predictions-for-en_bootstrap.jsonl --bootstrap 100


IoU: 0.11602812 ± 0.00977572
rho: -0.00234387 ± 0.02055886


In [ ]:
##RUN SCORER on VALIDATION SET and BASELINE_PREDICTIONS (no bootstrap)
!python {scorer} {mushroom} baseline_random-predictions-for-en.jsonl scores.txt

!cat scores.txt

IoU: 0.10600040
Cor: -0.00031311


In [24]:
my_predictions_gpt4o_mini = "/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_outputs/mushroom.en-val.v2.unlabeled.labelled_with_gpt-4o-mini_no_extra_keys_soft_labels_prob1.jsonl"

In [25]:
my_predictions_gpt4o_mini_v2 = "/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_outputs/mushroom.en-val.v2.unlabeled.labelled_with_gpt-4o-mini_no_extra_keys_soft_labels_prob1_v2.jsonl"

In [28]:
my_predictions_gemini = "/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_outputs/mushroom.en-val.v2.unlabeled.labelled_with_gemini-1.5-flash_no_extra_keys_soft_labels_prob1.jsonl"

In [26]:
results = os.path.join("/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/test_results/", "test_scores_results.txt")

In [27]:
my_predictions_gpt4o = "/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_outputs/mushroom.en-val.v2.unlabeled.labelled_with_gpt-4o-2024-08-06_no_extra_keys_soft_labels_prob1.jsonl"

In [37]:
my_predictions_gpt4o_v2 = "/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_outputs/mushroom.en-val.v2.unlabeled.labelled_with_gpt-4o-2024-08-06_no_extra_keys_soft_labels_prob1_v2.jsonl"

Below is a class method to unite/intersects spans of various predictions

In [ ]:
import json

class SpanProcessor:
    def __init__(self, file_paths, mode='union'):
        """
        Initialize the SpanProcessor with file paths and processing mode.

        :param file_paths: List of paths to the input JSONL files.
        :param mode: Processing mode, either 'union' or 'intersection'.
        """
        self.file_paths = file_paths
        self.mode = mode
        self.datasets = self._load_datasets()

    def _load_datasets(self):
        """Load datasets from JSONL files."""
        return [self._read_jsonl(file) for file in self.file_paths]

    @staticmethod
    def _read_jsonl(file_path):
        """Read a JSONL file."""
        with open(file_path, 'r', encoding='utf-8') as f:
            return [json.loads(line) for line in f]

    @staticmethod
    def _write_jsonl(data, file_path):
        """Write data to a JSONL file."""
        with open(file_path, 'w', encoding='utf-8') as f:
            for item in data:
                f.write(json.dumps(item) + '\n')

    @staticmethod
    def _merge_overlapping_spans(spans):
        """Merge overlapping or contiguous spans for hard labels."""
        if not spans:
            return []
        spans = sorted(spans, key=lambda x: x[0])  # Sort by start index
        merged = [spans[0]]
        for current in spans[1:]:
            prev = merged[-1]
            if current[0] <= prev[1]:  # Overlapping or contiguous
                merged[-1] = [prev[0], max(prev[1], current[1])]
            else:
                merged.append(current)
        return merged

    @staticmethod
    def _merge_soft_labels_with_avg_prob(soft_labels):
        """Merge overlapping or contiguous spans for soft labels with averaged probabilities."""
        if not soft_labels:
            return []

        sorted_labels = sorted(soft_labels, key=lambda x: x['start'])
        merged = [sorted_labels[0]]

        for current in sorted_labels[1:]:
            prev = merged[-1]
            if current['start'] <= prev['end']:  # Overlapping or contiguous
                total_length = (prev['end'] - prev['start']) + (current['end'] - current['start'])
                weighted_prob = (
                    (prev['prob'] * (prev['end'] - prev['start']) +
                     current['prob'] * (current['end'] - current['start']))
                    / total_length
                )
                # Clamp probability between 0 and 1
                weighted_prob = max(0.0, min(1.0, weighted_prob))
                merged[-1] = {
                    "start": prev['start'],
                    "end": max(prev['end'], current['end']),
                    "prob": weighted_prob
                }
            else:
                merged.append(current)

        return merged

    @staticmethod
    def _get_intersection_spans(spans1, spans2):
        """Find intersections between two sets of hard label spans."""
        intersections = []
        for span1 in spans1:
            for span2 in spans2:
                start = max(span1[0], span2[0])
                end = min(span1[1], span2[1])
                if start < end:  # Valid intersection
                    intersections.append([start, end])
        return intersections

    @staticmethod
    def _intersect_soft_labels_with_avg_prob(labels1, labels2):
        """Find intersections between two sets of soft labels with averaged probabilities, incorporating overlap length."""
        intersections = []
        # Number of experts (labels)
        num_experts = 3  # Adjust this if there are more experts (e.g., 3 for 3 experts)

        for span1 in labels1:
            for span2 in labels2:
                start = max(span1['start'], span2['start'])
                end = min(span1['end'], span2['end'])

                if start < end:  # Valid intersection
                  ## May be no need to use overlap_lenght.
                   # overlap_length = end - start

                    # Calculate weighted probability by averaging contributions of each expert
                    weighted_prob = (
                        (span1['prob'] + span2['prob']) / num_experts
                    )

                    # Adjust the probability by the overlap length to give more weight to longer overlaps
                   # weighted_prob *= overlap_length

                    # Normalize the probability between 0 and 1
                    weighted_prob = max(0.0, min(1.0, weighted_prob))

                    intersections.append({
                        "start": start,
                        "end": end,
                        "prob": weighted_prob,
                        #"overlap_length": overlap_length  # Include overlap length in the result
                    })

        return intersections


    def process(self):
      """Process datasets based on the selected mode ('union' or 'intersection')."""
      id_to_entries = {}
      for dataset in self.datasets:
          for entry in dataset:
              entry_id = entry['id']
              if entry_id not in id_to_entries:
                  id_to_entries[entry_id] = []
              id_to_entries[entry_id].append(entry)

      results = []
      for entry_id, entries in id_to_entries.items():
          base_entry = entries[0]
          combined_hard_labels = base_entry.get('hard_labels', None)
          combined_soft_labels = base_entry.get('soft_labels', [])

          for other_entry in entries[1:]:
              if self.mode == 'union':
                  if combined_hard_labels is not None:
                      combined_hard_labels = self._merge_overlapping_spans(
                          combined_hard_labels + other_entry.get('hard_labels', []))
                  combined_soft_labels = self._merge_soft_labels_with_avg_prob(
                      combined_soft_labels + other_entry.get('soft_labels', []))
              elif self.mode == 'intersection':
                  if combined_hard_labels is not None:
                      combined_hard_labels = self._get_intersection_spans(
                          combined_hard_labels, other_entry.get('hard_labels', []))
                  combined_soft_labels = self._intersect_soft_labels_with_avg_prob(
                      combined_soft_labels, other_entry.get('soft_labels', []))

          # Only include 'hard_labels' if they existed in the original entry
          if combined_hard_labels is not None:
              base_entry['hard_labels'] = combined_hard_labels

          base_entry['soft_labels'] = combined_soft_labels
          results.append(base_entry)

      return results

    def save_results(self, output_file):
        """Save processed results to a file."""
        results = self.process()
        self._write_jsonl(results, output_file)


In [ ]:
import json

## This function is to remove hard_labels from jsons. (needed for intersection)
def remove_hard_labels(input_file, key_to_remove='hard_labels'):
    """
    Processes a JSONL file by removing a specified key from each JSON object.

    Args:
        input_file (str): Path to the input JSONL file.
        key_to_remove (str): Key to remove from each JSON object.

    Returns:
        str: Path to the output JSONL file.
    """
    output_file = str(input_file).replace(".jsonl", "")+'_no_hard_labels.jsonl'

    # Open the input file, process each line, and save to the output file
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        for line in infile:
            try:
                # Parse the JSON line
                data = json.loads(line)

                # Remove the specified key if it exists
                if key_to_remove in data:
                    del data[key_to_remove]

                # Write the updated line back to the output file
                outfile.write(json.dumps(data) + '\n')
            except json.JSONDecodeError:
                print(f"Skipping invalid JSON line: {line.strip()}")

    #print(f"Processed lines from {input_file} and saved to {output_file}.")

    return output_file


In [ ]:
!cp {my_predictions_gemini} ./my_predictions_gemini

In [ ]:
file4 =  remove_hard_labels("/content/my_predictions_gemini")
print(file4)

/content/my_predictions_gemini_no_hard_labels.jsonl


In [ ]:
!cp {my_predictions_gpt4o_mini} ./my_predictions_gpt4o_mini

In [ ]:
file5 =  remove_hard_labels("/content/my_predictions_gpt4o_mini")
print(file5)

/content/my_predictions_gpt4o_mini_no_hard_labels.jsonl


In [ ]:
file6 = "/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/results/bert_zang_100_results.jsonl"

In [ ]:
file7 = "/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/results/bert_nick_100_results.jsonl"

In [ ]:
file8 = "/content/pred_rule_based_ner.jsonl"

In [ ]:
file9 = remove_hard_labels(file8)
print(file9)

FileNotFoundError: [Errno 2] No such file or directory: '/content/pred_rule_based_ner.jsonl'

In [ ]:
print(file9)

NameError: name 'file9' is not defined

In [ ]:
#this is gemini with no hard labels
!python {scorer} {mushroom} {file4} my_prediction_scores.txt
!cat my_prediction_scores.txt

IoU: 0.39023997
Cor: 0.40379501


In [ ]:
!python {scorer} {mushroom} /content/my_predictions_gemini my_prediction_scores.txt
!cat my_prediction_scores.txt

IoU: 0.39023997
Cor: 0.40379501


In [ ]:
#this is gpt-4o-mini with no hard labels
!python {scorer} {mushroom} {file5} my_prediction_scores.txt
!cat my_prediction_scores.txt

IoU: 0.39866628
Cor: 0.44332700


In [ ]:
#this is gpt-4o-mini with hard labels
!python {scorer} {mushroom} /content/my_predictions_gpt4o_mini my_prediction_scores.txt
!cat my_prediction_scores.txt

IoU: 0.39866628
Cor: 0.44332700


In [ ]:
!python {scorer} {mushroom} {file6} my_prediction_scores.txt
!cat my_prediction_scores.txt

IoU: 0.04000000
Cor: 0.00000000


In [ ]:
!python {scorer} {mushroom} {file7} my_prediction_scores.txt
!cat my_prediction_scores.txt

IoU: 0.04000000
Cor: 0.00630395


In [30]:
!python {scorer} {mushroom} {file8} my_prediction_scores.txt
!cat my_prediction_scores.txt

python3: can't open file '/content/{scorer}': [Errno 2] No such file or directory
IoU: 0.36537537
Cor: 0.36843457


In [ ]:
#same as file8
!python {scorer} {mushroom} {file9} my_prediction_scores.txt
!cat my_prediction_scores.txt

python3: can't open file '/content/{scorer}': [Errno 2] No such file or directory
IoU: 0.04000000
Cor: 0.00630395


In [32]:
!python /content/scorer.py /content/mushroom.en-val.v2.jsonl {my_predictions_gpt4o_v2} my_prediction_scores.txt
!cat my_prediction_scores.txt

/content/scorer.py:40: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_json(filename, lines=True)
usage: scorer.py [-h] ref_file pred_file output_file
scorer.py: error: argument pred_file: invalid <lambda> value: '{my_predictions_gpt4o_v2}'
IoU: 0.36537537
Cor: 0.36843457


In [33]:
!python {scorer} {mushroom} {my_predictions_gpt4o_mini} my_prediction_scores.txt
!cat my_prediction_scores.txt

IoU: 0.39866628
Cor: 0.44332700


In [34]:
!python {scorer} {mushroom} {my_predictions_gpt4o_mini_v2} my_prediction_scores.txt
!cat my_prediction_scores.txt

IoU: 0.36537537
Cor: 0.36843457


In [36]:
!python {scorer} {mushroom} {my_predictions_gpt4o} my_prediction_scores.txt
!cat my_prediction_scores.txt

IoU: 0.38394451
Cor: 0.39184584


In [38]:
!python {scorer} {mushroom} {my_predictions_gpt4o_v2} my_prediction_scores.txt
!cat my_prediction_scores.txt

IoU: 0.34869890
Cor: 0.39082698


In [ ]:
# Intesection/Union
file_paths = [file4,
              file5, # Add more files as needed [ file1, file2, file3] etc...
              file8,
              my_predictions_gpt4o
              ]
modes = ['union', 'intersection']
output_file = f'/content/output_{mode}_LLMs_rule-based.jsonl'

for mode in modes:
  output_file = f'/content/output_{mode}_LLMs_rule-based.jsonl'
  processor = SpanProcessor(file_paths, mode)
  processor.save_results(output_file)



NameError: name 'mode' is not defined

In [ ]:
intersection_file = "/content/output_intersection_LLMs_rule-based.jsonl"

In [ ]:
union_file = "/content/output_union_LLMs_rule-based.jsonl"

In [ ]:
!echo "INTERSECTION LLMS ONLY"
!python {scorer} {mushroom} {intersection_file} my_prediction_scores.txt
!cat my_prediction_scores.txt

INTERSECTION LLMS ONLY
/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/participant_kit/scorer.py:40: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_json(filename, lines=True)
usage: scorer.py [-h] ref_file pred_file output_file
scorer.py: error: argument pred_file: invalid <lambda> value: '/content/output_intersection_LLMs_rule-based.jsonl'
IoU: 0.04000000
Cor: 0.00630395


In [ ]:
!echo "UNION LLMS ONLY"
!python {scorer} {mushroom} {union_file} my_prediction_scores.txt
!cat my_prediction_scores.txt

UNION LLMS ONLY
/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/participant_kit/scorer.py:40: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_json(filename, lines=True)
usage: scorer.py [-h] ref_file pred_file output_file
scorer.py: error: argument pred_file: invalid <lambda> value: '/content/output_union_LLMs_rule-based.jsonl'
IoU: 0.04000000
Cor: 0.00630395


In [ ]:
# Function to load the .jsonl file and parse its contents
def load_jsonl(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        # Read each line and parse as a JSON object
        return [json.loads(line) for line in file]

# Load the data
data = load_jsonl(file8)


In [ ]:
for k in data[0].keys():
    print(k)

id
model_input
model_output_text
soft_labels
